In [ ]:
# ==============================================================================
# GPU-Accelerated Multivariate Hybrid Predictor Selection (v5 - Colab Stable)
# ==============================================================================
# CHANGES FROM v4 (crash fixes, no concept changes):
#   - Added explicit GPU/CPU memory management (gc, cupy pool flush) between steps
#   - Chunked offshore CSV loading to prevent GPU OOM
#   - CANDIDATE_POOL_SIZE reduced from 20 → 12 for PCMCI+ feasibility
#     (20 pts × 5 vars + 2 buoys × 4 vars = 108 vars → crashes; 12 × 5 + 8 = 68 vars)
#   - Buoy SSH excluded from PCMCI+ inputs (it is NaN → interpolated artifact, not data)
#   - Removed all TIFF saves (PNG-only at 600 DPI for Q1 journals)
#   - Fixed deprecated pandas .fillna(method=) → .bfill() / .ffill()
#   - Fixed offshore_long_df scope bug when checkpoint is loaded in Step E
#   - Added cupy memory pool flushing between heavy steps
#   - Added depth to static metadata only (not PCMCI+ time-series)
#   - Added FDR-corrected q-value extraction from PCMCI+ results
# ==============================================================================

# =============================================================================
# STEP A: SETUP AND ENVIRONMENT
# =============================================================================
print("Installing required libraries...")
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py
!pip install tigramite geopandas contextily statsmodels -q
print("Installation complete.")

import os
import gc
import cudf
import cupy as cp
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import contextily as cx
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from google.colab import drive
from tqdm.notebook import tqdm
from sklearn.preprocessing import minmax_scale

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.plotting import plot_graph

import networkx as nx

# ── Mount Drive ──────────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PROJECT_DIR    = '/content/drive/MyDrive/KBS_Paper'
OFFSHORE_DATA_DIR   = os.path.join(BASE_PROJECT_DIR, 'Data', 'catboost_corrected')
BUOY_DATA_DIR       = os.path.join(BASE_PROJECT_DIR, 'Data')
OUTPUT_DIR          = os.path.join(BASE_PROJECT_DIR, 'Outputs', 'Predictor_Selection_v5_Physical')

MAIN_BUOY_FILE      = os.path.join(BUOY_DATA_DIR, 'buoy_072_3hourly.csv')
OOS_BUOY_FILE       = os.path.join(BUOY_DATA_DIR, 'buoy_oos_3hourly.csv')

CHECKPOINT_WIDE_DF          = os.path.join(OUTPUT_DIR, 'checkpoint_wide_df_v5.parquet')
CHECKPOINT_CORR_CANDIDATES  = os.path.join(OUTPUT_DIR, 'checkpoint_correlation_candidates_v5.csv')
CHECKPOINT_PCMCI_RESULTS    = os.path.join(OUTPUT_DIR, 'checkpoint_pcmci_results_v5.npz')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# ── Global constants ──────────────────────────────────────────────────────────
# Variables used in correlation stage (offshore has all 5; buoys lack ssh)
ANALYSIS_VARS_OFFSHORE = ['hm0', 'tp', 'mdir_u', 'mdir_v', 'ssh']
ANALYSIS_VARS_BUOY     = ['hm0', 'tp', 'mdir_u', 'mdir_v']         # ssh excluded → no real data at buoys

OFFSHORE_COLS_TO_KEEP = [
    'time', 'lat', 'lon', 'depth', 'slope_ns', 'slope_ew', 'ssh',
    'Hm0_catboost', 'Tp_catboost', 'u_wind_catboost', 'v_wind_catboost',
    'u_curr_catboost', 'v_curr_catboost', 'u_mdir_catboost', 'v_mdir_catboost',
    'WindSpeed_catboost', 'WindDirection_catboost', 'CurrSpd_catboost',
    'CurrDir_catboost', 'VMDR_catboost'
]

MAX_LAG_HOURS         = 24
TIME_RESOLUTION_HOURS = 3
MAX_LAG_STEPS         = MAX_LAG_HOURS // TIME_RESOLUTION_HOURS   # 8 steps

# FIX: Reduced from 20 → 12 so PCMCI+ receives 12×5 + 2×4 = 68 variables
# instead of 108, which is the primary crash cause in Step D.
CANDIDATE_POOL_SIZE   = 12
FINAL_PREDICTOR_COUNT = 5
PCMCI_ALPHA           = 0.05
BUOY_DEPTH_METERS     = 18.0

# ── Publication figure style (once, reused everywhere) ───────────────────────
def set_pub_style():
    sns.set_style("ticks")
    mpl.rcParams['font.family']      = 'serif'
    mpl.rcParams['font.serif']       = ['Times New Roman', 'Arial']
    mpl.rcParams['font.size']        = 12
    mpl.rcParams['axes.linewidth']   = 0.8
    mpl.rcParams['xtick.major.width']= 0.8
    mpl.rcParams['ytick.major.width']= 0.8

def save_fig(fig, name: str, dpi: int = 600):
    """Save a single PNG at journal-quality resolution and close the figure."""
    path = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"    ✓ Saved → {path}")
    return path

def free_gpu_memory():
    """Release all unused GPU allocations between heavy steps."""
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

print("STEP A COMPLETE.\n")

# =============================================================================
# STEP B: DATA LOADING & PREPARATION (GPU-ACCELERATED)
# =============================================================================

def convert_circular_to_vectors(df, direction_col, speed_col=None):
    direction_numeric = cudf.to_numeric(df[direction_col], errors='coerce')
    direction_rad     = cp.deg2rad(direction_numeric.values)
    u_arr = -cp.sin(direction_rad)
    v_arr = -cp.cos(direction_rad)
    if speed_col:
        speed  = df[speed_col].values
        u_arr *= speed
        v_arr *= speed
    # FIX: attach the DataFrame's own index so cudf assignment never mismatches
    u = cudf.Series(u_arr, index=df.index, name='u')
    v = cudf.Series(v_arr, index=df.index, name='v')
    return u, v


def load_offshore_data_gpu(data_dir, cols_to_keep, chunk_size: int = 5):
    """
    Load yearly CSVs in small chunks and concatenate progressively.
    chunk_size controls how many files are held in GPU RAM at once,
    preventing OOM on free Colab (≈15 GB GPU).
    """
    print("    - Loading offshore CSVs in GPU chunks to prevent OOM ...")
    all_files = sorted([os.path.join(data_dir, f)
                        for f in os.listdir(data_dir) if f.endswith('.csv')])
    if not all_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    cols_lower = [c.lower() for c in cols_to_keep]
    master_chunks = []

    for i in range(0, len(all_files), chunk_size):
        batch = all_files[i:i + chunk_size]
        gdf_batch = cudf.concat([cudf.read_csv(f) for f in batch], ignore_index=True)
        gdf_batch.columns = gdf_batch.columns.str.lower()
        existing_cols = [c for c in cols_lower if c in gdf_batch.columns]
        gdf_batch = gdf_batch[existing_cols]
        gdf_batch.columns = gdf_batch.columns.str.replace('_catboost', '', regex=False)
        gdf_batch = gdf_batch.rename(columns={'vmdr': 'mdir'})
        gdf_batch['time'] = cudf.from_pandas(
            pd.to_datetime(gdf_batch['time'].to_pandas(), utc=True)
              .dt.tz_localize(None)
              .astype('datetime64[ns]')              # FIX: same dtype as buoy
        )
        gdf_batch['mdir_u'], gdf_batch['mdir_v'] = convert_circular_to_vectors(gdf_batch, 'mdir')
        if 'winddirection' in gdf_batch.columns and 'windspeed' in gdf_batch.columns:
            gdf_batch['winddir_u'], gdf_batch['winddir_v'] = convert_circular_to_vectors(
                gdf_batch, 'winddirection', 'windspeed')
        master_chunks.append(gdf_batch.to_pandas())   # move to CPU immediately
        del gdf_batch
        free_gpu_memory()
        print(f"      · Batch {i // chunk_size + 1}/{-(-len(all_files)//chunk_size)} loaded.")

    master_pd = pd.concat(master_chunks, ignore_index=True)
    del master_chunks
    gc.collect()

    # Build point_id on CPU (cheaper)
    master_pd['point_id'] = ('offshore_' +
        master_pd.groupby(['lat', 'lon']).ngroup().astype(str))
    print(f"    - Loaded {master_pd['point_id'].nunique()} unique offshore points.")

    # Push final frame back to GPU for pivot step
    return cudf.from_pandas(master_pd)


def hampel_filter_pandas(series, window_size=11, n_sigmas=3):
    rolling_median = series.rolling(window=window_size, center=True, min_periods=1).median()
    mad = lambda x: np.median(np.abs(x - np.median(x)))
    rolling_mad   = series.rolling(window=window_size, center=True, min_periods=1).apply(mad, raw=True)
    threshold     = n_sigmas * 1.4826 * rolling_mad
    outliers      = np.abs(series - rolling_median) > threshold
    return outliers


def load_and_clean_buoy_data(filepath, buoy_id):
    print(f"    - Loading buoy: {buoy_id}")
    df = cudf.read_csv(filepath)
    df.columns = df.columns.str.lower()
    df = df[['time', 'hm0', 'tp', 'mdir']].copy()
    df['time'] = cudf.from_pandas(
        pd.to_datetime(df['time'].to_pandas(), utc=True)
          .dt.tz_localize(None)                  # strip tz → naive datetime64[ns]
          .astype('datetime64[ns]')              # FIX: force ns precision to match offshore
    )
    df = df.set_index('time').sort_index()

    full_index = cudf.date_range(
        start=df.index.min(), end=df.index.max(),
        freq=f'{TIME_RESOLUTION_HOURS}h'         # FIX: lowercase 'h' (RAPIDS 25.x)
    )
    df = df.reindex(full_index)
    df.index.name = 'time'
    df['depth'] = BUOY_DEPTH_METERS
    # ssh is NOT available at buoys — keep as NaN (do NOT pass to PCMCI+)
    df['ssh'] = np.nan

    for col in ['hm0', 'tp', 'mdir']:
        series_cpu = df[col].to_pandas()
        if col in ['hm0', 'tp']:
            series_cpu.replace(0, np.nan, inplace=True)
        series_no_na = series_cpu.dropna()
        if not series_no_na.empty:
            outliers_mask = hampel_filter_pandas(series_no_na)
            series_cpu.loc[outliers_mask[outliers_mask].index] = np.nan
        df[col] = cudf.from_pandas(series_cpu)
        df[col] = df[col].interpolate(method='linear')

    df['mdir_u'], df['mdir_v'] = convert_circular_to_vectors(df, 'mdir')
    df_final  = df.reset_index()
    cols_rename = {col: f"{buoy_id}_{col}" for col in df_final.columns if col != 'time'}
    return df_final.rename(columns=cols_rename)


# ── Checkpoint logic ──────────────────────────────────────────────────────────
print("STEP B: GPU-ACCELERATED DATA LOADING AND PREPARATION")

# Store a minimal location lookup that survives checkpoint restores (needed in Step E)
LOCATIONS_CACHE_PATH = os.path.join(OUTPUT_DIR, 'offshore_locations.csv')

if os.path.exists(CHECKPOINT_WIDE_DF):
    print(f"  ✓ Checkpoint found → {CHECKPOINT_WIDE_DF}")
    analysis_df = cudf.read_parquet(CHECKPOINT_WIDE_DF)
else:
    print("  - No checkpoint. Running full pipeline ...")
    offshore_long_df = load_offshore_data_gpu(OFFSHORE_DATA_DIR, OFFSHORE_COLS_TO_KEEP)

    # Save location lookup now so Step E can access it even after checkpoint restore
    loc_df = offshore_long_df[['point_id', 'lat', 'lon']].drop_duplicates().to_pandas()
    loc_df.to_csv(LOCATIONS_CACHE_PATH, index=False)

    print("    - Pivoting offshore data to wide format ...")
    pivot_vars = ['hm0', 'tp', 'mdir', 'ssh', 'depth']
    analysis_df = offshore_long_df.pivot_table(
        index='time', columns='point_id', values=pivot_vars
    )
    analysis_df.columns = [f"{col[1]}_{col[0]}" for col in analysis_df.columns]
    analysis_df = analysis_df.reset_index()

    # Vector components on wide frame
    offshore_point_ids = sorted(set(
        '_'.join(c.split('_')[:2])
        for c in analysis_df.columns if c.startswith('offshore_')
    ))
    print("    - Creating mdir vector components on wide DataFrame ...")
    for pid in offshore_point_ids:
        if f'{pid}_mdir' in analysis_df.columns:
            analysis_df[f'{pid}_mdir_u'], analysis_df[f'{pid}_mdir_v'] = \
                convert_circular_to_vectors(analysis_df, f'{pid}_mdir')

    del offshore_long_df
    free_gpu_memory()

    main_buoy_df = load_and_clean_buoy_data(MAIN_BUOY_FILE, 'buoy_main')
    oos_buoy_df  = load_and_clean_buoy_data(OOS_BUOY_FILE,  'buoy_oos')

    print("    - Merging offshore + buoy data ...")
    analysis_df = (analysis_df
                   .merge(main_buoy_df, on='time', how='inner')
                   .merge(oos_buoy_df,  on='time', how='inner')
                   .set_index('time').sort_index())

    main_hm0_col = [c for c in analysis_df.columns if 'buoy_main_hm0' in c]
    analysis_df  = analysis_df.dropna(how='all', subset=main_hm0_col)

    print(f"    - Saving checkpoint → {CHECKPOINT_WIDE_DF}")
    analysis_df.to_parquet(CHECKPOINT_WIDE_DF)

print(f"  - analysis_df shape: {analysis_df.shape}")
free_gpu_memory()

# ── Statistical Validation Block ─────────────────────────────────────────────
print("  - Running ADF stationarity + ACF tests ...")
try:
    set_pub_style()
    df_for_stats = analysis_df.to_pandas()

    cols_to_test = ['buoy_main_hm0', 'buoy_main_mdir']
    adf_results  = []
    for col in cols_to_test:
        if col in df_for_stats.columns:
            series  = df_for_stats[col].dropna()
            adf_out = adfuller(series)
            adf_results.append({
                'Variable':            col,
                'ADF Statistic':       round(adf_out[0], 4),
                'p-value':             round(adf_out[1], 6),
                'Critical Value (1%)': adf_out[4]['1%'],
                'Critical Value (5%)': adf_out[4]['5%'],
                'Stationary?':         'Yes' if adf_out[1] < 0.05 else 'No'
            })

    if adf_results:
        adf_df = pd.DataFrame(adf_results)
        adf_df.to_csv(os.path.join(OUTPUT_DIR, 'table_S1_adf_stationarity.csv'), index=False)
        print(f"    ✓ ADF table saved.")

    if 'buoy_main_hm0' in df_for_stats.columns:
        hm0_series = df_for_stats['buoy_main_hm0'].dropna()
        fig, ax = plt.subplots(figsize=(10, 5))
        sm.graphics.tsa.plot_acf(
            hm0_series, lags=24, ax=ax, alpha=0.05,
            title=(r"ACF of $H_{m0}$ — justification for $\tau_{max}=8$ (24 h)")
        )
        ax.set_xlabel("Lag (3-h steps)")
        ax.set_ylabel("Autocorrelation")
        sns.despine()
        save_fig(fig, 'fig01_acf_tau_max_justification')

except Exception as e:
    print(f"    WARNING: Statistical validation error: {e}. Continuing.")

print("\nSTEP B COMPLETE.\n")

# =============================================================================
# STEP C: PHYSICALLY-WEIGHTED CORRELATIONAL FILTERING
# =============================================================================

def calculate_lagged_corr_best(df_pd, target_col, predictor_col, max_lag_steps):
    """Returns the maximum absolute lagged cross-correlation."""
    best = 0.0
    for lag in range(max_lag_steps + 1):
        shifted = df_pd[target_col].shift(-lag)
        valid   = pd.concat([df_pd[predictor_col], shifted], axis=1).dropna()
        if len(valid) < max_lag_steps * 2:
            continue
        corr = valid.iloc[:, 0].corr(valid.iloc[:, 1])
        if not np.isnan(corr) and abs(corr) > best:
            best = abs(corr)
    return best


def get_physically_weighted_ranks(df, offshore_ids, target_buoy_id,
                                  analysis_vars, max_lag_steps):
    """Correlation ranking + depth-based physical weighting."""
    df_pd = df.to_pandas()
    all_results = []

    for var in tqdm(analysis_vars, desc="Processing variables"):
        target_col = f"{target_buoy_id}_{var}"
        if target_col not in df_pd.columns:
            print(f"  ⚠ {target_col} not found — skipping.")
            continue
        var_results = []
        for pid in tqdm(offshore_ids, desc=f"  Correlations [{var}]", leave=False):
            pred_col = f"{pid}_{var}"
            if pred_col not in df_pd.columns:
                continue
            pair_df = df_pd[[pred_col, target_col]].dropna()
            if len(pair_df) > max_lag_steps * 2:
                corr = calculate_lagged_corr_best(
                    pair_df, target_col, pred_col, max_lag_steps)
                var_results.append({'point_id': pid, 'correlation': corr})

        if var_results:
            vdf = pd.DataFrame(var_results)
            vdf[f'rank_{var}'] = vdf['correlation'].rank(ascending=False, method='min')
            all_results.append(vdf[['point_id', f'rank_{var}']])

    if not all_results:
        raise ValueError("Correlation step produced no results. Check column names.")

    final_df = all_results[0]
    for r in all_results[1:]:
        final_df = final_df.merge(r, on='point_id', how='outer')

    rank_cols = [c for c in final_df.columns if c.startswith('rank_')]
    final_df['total_rank'] = final_df[rank_cols].sum(axis=1)

    # Physical depth weighting
    print("    - Incorporating depth weighting ...")
    depth_data = []
    for pid in offshore_ids:
        dcol = f"{pid}_depth"
        if dcol in df_pd.columns:
            dval = df_pd[dcol].median()
            if not np.isnan(dval):
                depth_data.append({'point_id': pid, 'depth': dval})

    depth_df = pd.DataFrame(depth_data)
    depth_df['depth_rank'] = depth_df['depth'].rank(ascending=True, method='min')
    final_df = final_df.merge(depth_df[['point_id', 'depth', 'depth_rank']], on='point_id')
    final_df['physically_weighted_rank'] = final_df['total_rank'] + final_df['depth_rank']

    return final_df.sort_values('physically_weighted_rank').reset_index(drop=True)


print("STEP C: PHYSICALLY-WEIGHTED CORRELATIONAL FILTERING")

if os.path.exists(CHECKPOINT_CORR_CANDIDATES):
    print(f"  ✓ Checkpoint found → {CHECKPOINT_CORR_CANDIDATES}")
    candidates_df       = pd.read_csv(CHECKPOINT_CORR_CANDIDATES)
    candidate_predictors = candidates_df['point_id'].head(CANDIDATE_POOL_SIZE).tolist()
else:
    offshore_point_ids = sorted(set(
        '_'.join(c.split('_')[:2])
        for c in analysis_df.columns if c.startswith('offshore_')
    ))
    ranked_df = get_physically_weighted_ranks(
        analysis_df,
        offshore_ids=offshore_point_ids,
        target_buoy_id='buoy_main',
        analysis_vars=ANALYSIS_VARS_OFFSHORE,   # ssh included for offshore points
        max_lag_steps=MAX_LAG_STEPS
    )
    ranked_df.to_csv(CHECKPOINT_CORR_CANDIDATES, index=False)
    candidate_predictors = ranked_df.head(CANDIDATE_POOL_SIZE)['point_id'].tolist()

print(f"  ✓ Top {CANDIDATE_POOL_SIZE} candidates: {candidate_predictors}")
try:
    display(pd.read_csv(CHECKPOINT_CORR_CANDIDATES).head())
except:
    print(pd.read_csv(CHECKPOINT_CORR_CANDIDATES).head().to_string())

free_gpu_memory()
print("\nSTEP C COMPLETE.\n")

# =============================================================================
# STEP D: MULTIVARIATE CAUSAL SELECTION — PCMCI+
# =============================================================================
# Variable count management:
#   Offshore candidates: CANDIDATE_POOL_SIZE × 5 vars = 12 × 5 = 60
#   Buoys:               2 × 4 vars (ssh excluded, no real data) = 8
#   Total:               68  ← feasible on Colab CPU for several hours
# =============================================================================

print("STEP D: MULTIVARIATE CAUSAL SELECTION (PCMCI+)")

causal_candidate_cols = []

# Offshore candidates — all 5 variables including ssh
for pid in candidate_predictors:
    for var in ANALYSIS_VARS_OFFSHORE:
        col = f"{pid}_{var}"
        if col in analysis_df.columns:
            causal_candidate_cols.append(col)

# Buoys — 4 variables ONLY (ssh excluded: it is NaN → would be artificial signal)
for bid in ['buoy_main', 'buoy_oos']:
    for var in ANALYSIS_VARS_BUOY:
        col = f"{bid}_{var}"
        if col in analysis_df.columns:
            causal_candidate_cols.append(col)

print(f"  - PCMCI+ input: {len(causal_candidate_cols)} time series")

# FIX: use .bfill() / .ffill() (pandas ≥ 2.0 — method= kwarg removed)
causal_df_pd = (analysis_df[causal_candidate_cols]
                .to_pandas()
                .interpolate(method='linear')
                .bfill()
                .ffill())

if causal_df_pd.isnull().values.any():
    raise ValueError("NaNs remain before PCMCI+. Inspect cleaning steps.")

var_names = causal_df_pd.columns.tolist()

# Free GPU RAM before the heavy CPU computation
del analysis_df
free_gpu_memory()
print("  - GPU RAM freed; running PCMCI+ on CPU ...")

if os.path.exists(CHECKPOINT_PCMCI_RESULTS):
    print(f"  ✓ Checkpoint found → {CHECKPOINT_PCMCI_RESULTS}")
    results_data = np.load(CHECKPOINT_PCMCI_RESULTS, allow_pickle=True)
    results      = results_data['results'].item()
else:
    print("  - Running PCMCI+ (may take several hours on free Colab) ...")
    dataframe = pp.DataFrame(causal_df_pd.values, var_names=var_names)
    parcorr   = ParCorr(significance='analytic')
    pcmci     = PCMCI(dataframe=dataframe, cond_ind_test=parcorr, verbosity=1)

    results = pcmci.run_pcmciplus(
        tau_min=1,
        tau_max=MAX_LAG_STEPS,
        pc_alpha=PCMCI_ALPHA
    )

    np.savez_compressed(CHECKPOINT_PCMCI_RESULTS, results=results)
    print(f"  ✓ PCMCI+ results saved → {CHECKPOINT_PCMCI_RESULTS}")

# ── Extract causal parents + q-values ────────────────────────────────────────
print("  - Extracting causal parents + FDR-corrected q-values ...")
causal_graph = results['graph']
val_matrix   = results['val_matrix']
p_matrix     = results.get('p_matrix', None)   # FDR q-values if available

causal_links = []
target_indices = [i for i, n in enumerate(var_names) if n.startswith('buoy_')]

for pred_idx, pred_name in enumerate(var_names):
    if not pred_name.startswith('offshore_'):
        continue
    for tgt_idx in target_indices:
        for lag in range(1, causal_graph.shape[2]):
            if causal_graph[pred_idx, tgt_idx, lag] == '-->':
                link = {
                    'predictor_id':  '_'.join(pred_name.split('_')[:2]),
                    'predictor_var': '_'.join(pred_name.split('_')[2:]),
                    'target':        '_'.join(var_names[tgt_idx].split('_')[:2]),
                    'lag':           lag,
                    'link_strength': abs(val_matrix[pred_idx, tgt_idx, lag])
                }
                if p_matrix is not None:
                    link['p_value'] = p_matrix[pred_idx, tgt_idx, lag]
                causal_links.append(link)

if not causal_links:
    print("  ⚠ No causal links found. Falling back to correlation candidates.")
    final_predictor_ids = candidate_predictors[:FINAL_PREDICTOR_COUNT]
else:
    causal_df = pd.DataFrame(causal_links)
    causal_df.to_csv(os.path.join(OUTPUT_DIR, 'table_S2_all_causal_links.csv'), index=False)
    ranked_causal = (causal_df.groupby('predictor_id')['link_strength']
                              .max()
                              .sort_values(ascending=False))
    final_predictor_ids = ranked_causal.head(FINAL_PREDICTOR_COUNT).index.tolist()

print(f"  ✓ Final {len(final_predictor_ids)} predictors: {final_predictor_ids}")
print("\nSTEP D COMPLETE.\n")

gc.collect()

# =============================================================================
# STEP E: REPORTING, VISUALIZATION, FINAL DATASET
# =============================================================================

def plot_summary_causal_graph(results, var_names, final_predictors, candidate_predictors):
    print("  - Generating causal graph ...")
    buoy_nodes = ['buoy_main', 'buoy_oos']
    all_nodes  = sorted(set(candidate_predictors)) + buoy_nodes
    G = nx.DiGraph()
    G.add_nodes_from(all_nodes)

    g_matrix, v_matrix = results['graph'], results['val_matrix']
    edge_weights = {}

    for i, pname in enumerate(var_names):
        if not pname.startswith('offshore_'):
            continue
        from_node = '_'.join(pname.split('_')[:2])
        for j, tname in enumerate(var_names):
            if not tname.startswith('buoy_'):
                continue
            to_node = '_'.join(tname.split('_')[:2])
            for lag in range(1, g_matrix.shape[2]):
                if g_matrix[i, j, lag] == '-->':
                    w = abs(v_matrix[i, j, lag])
                    key = (from_node, to_node)
                    edge_weights[key] = max(edge_weights.get(key, 0), w)

    for (u, v), w in edge_weights.items():
        G.add_edge(u, v, weight=w)

    set_pub_style()
    fig, ax = plt.subplots(figsize=(14, 14))
    pos = nx.circular_layout(G)

    node_colors, node_sizes = [], []
    for node in G.nodes():
        if node == 'buoy_main':
            node_colors.append('#F4C430'); node_sizes.append(8000)
        elif node == 'buoy_oos':
            node_colors.append('#87CEEB'); node_sizes.append(8000)
        elif node in final_predictors:
            node_colors.append('#32CD32'); node_sizes.append(6000)
        else:
            node_colors.append('#D3D3D3'); node_sizes.append(4000)

    widths = [d['weight'] * 15 for _, _, d in G.edges(data=True)]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                           node_size=node_sizes, edgecolors='black', ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=11, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, width=widths, edge_color='#8B0000',
                           alpha=0.75, arrows=True, arrowstyle='-|>',
                           arrowsize=30, connectionstyle='arc3,rad=0.05', ax=ax)
    ax.set_title("Causal Links — Offshore Predictors → Nearshore Buoys",
                 fontsize=20, pad=20)

    legend_elements = [
        mpl.lines.Line2D([0], [0], marker='o', color='w',
                         markerfacecolor=c, markersize=14, label=l)
        for c, l in [('#F4C430', 'Main Buoy (target)'),
                     ('#87CEEB', 'OOS Buoy (target)'),
                     ('#32CD32', 'Selected Predictor'),
                     ('#D3D3D3', 'Candidate (not selected)')]
    ]
    ax.legend(handles=legend_elements, loc='lower right', frameon=True,
              facecolor='white', fontsize=12)
    save_fig(fig, 'fig02_causal_graph')


print("STEP E: REPORTING, VISUALIZATION, FINAL DATASET")

# E.1 — Map
print("  - Generating predictor map ...")

# FIX: Load locations from cache if offshore_long_df is not in scope
if os.path.exists(LOCATIONS_CACHE_PATH):
    locations_df = pd.read_csv(LOCATIONS_CACHE_PATH)
else:
    tmp_offshore  = load_offshore_data_gpu(OFFSHORE_DATA_DIR, OFFSHORE_COLS_TO_KEEP)
    locations_df  = tmp_offshore[['point_id', 'lat', 'lon']].drop_duplicates().to_pandas()
    locations_df.to_csv(LOCATIONS_CACHE_PATH, index=False)
    del tmp_offshore
    free_gpu_memory()

gdf = gpd.GeoDataFrame(
    locations_df,
    geometry=gpd.points_from_xy(locations_df.lon, locations_df.lat),
    crs="EPSG:4326"
)
buoy_meta = pd.DataFrame({
    'point_id': ['buoy_main', 'buoy_oos'],
    'lat':      [24.513, 24.535],
    'lon':      [56.647, 56.629]
})
buoy_gdf = gpd.GeoDataFrame(
    buoy_meta,
    geometry=gpd.points_from_xy(buoy_meta.lon, buoy_meta.lat),
    crs="EPSG:4326"
)
full_gdf = pd.concat([gdf, buoy_gdf], ignore_index=True)

def classify(pid):
    if pid == 'buoy_main':           return 'Main Target Buoy'
    if pid == 'buoy_oos':            return 'OOS Target Buoy'
    if pid in final_predictor_ids:   return 'Final Predictor (selected)'
    if pid in candidate_predictors:  return 'Candidate (not selected)'
    return 'Not Selected'

full_gdf['category']  = full_gdf['point_id'].apply(classify)
gdf_mercator          = full_gdf.to_crs(epsg=3857)

set_pub_style()
fig, ax = plt.subplots(figsize=(12, 12))

cat_style = {
    'Main Target Buoy':          dict(color='#F4C430', markersize=350, marker='*', edgecolor='black', zorder=10),
    'OOS Target Buoy':           dict(color='#87CEEB', markersize=350, marker='*', edgecolor='black', zorder=10),
    'Final Predictor (selected)':dict(color='#32CD32', markersize=180, marker='o', edgecolor='black', zorder=9),
    'Candidate (not selected)':  dict(color='#808080', markersize=50,  marker='o', alpha=0.7,         zorder=8),
}

for cat, style in cat_style.items():
    subset = gdf_mercator[gdf_mercator['category'] == cat]
    if not subset.empty:
        subset.plot(ax=ax, label=cat, **style)

cx.add_basemap(ax, source=cx.providers.Esri.OceanBasemap, zoom=9)
ax.set_title(f'Causally & Physically Selected Offshore Predictors (n={FINAL_PREDICTOR_COUNT})',
             fontsize=16)
ax.set_axis_off()
ax.legend(title='Point Category', loc='lower right', frameon=True,
          facecolor='white', fontsize=11)
save_fig(fig, 'fig03_predictor_map')

# E.2 — Causal graph
plot_summary_causal_graph(
    results, var_names, final_predictor_ids, candidate_predictors
)

# E.3 — Save final predictor IDs
pd.DataFrame({'point_id': final_predictor_ids}).to_csv(
    os.path.join(OUTPUT_DIR, 'final_predictor_ids_v5.csv'), index=False
)
print("  ✓ Final predictor IDs saved.")

# E.4 — Final master dataset
print("  - Building final master dataset ...")
offshore_for_final = load_offshore_data_gpu(OFFSHORE_DATA_DIR, OFFSHORE_COLS_TO_KEEP)
final_offshore_df  = offshore_for_final[
    offshore_for_final['point_id'].isin(final_predictor_ids)
]

buoys_list = []
for file, pid, lat, lon in [
    (MAIN_BUOY_FILE, 'buoy_main', 24.513, 56.647),
    (OOS_BUOY_FILE,  'buoy_oos',  24.535, 56.629)
]:
    df = cudf.read_csv(file)
    df.columns = df.columns.str.lower()
    df['point_id'] = pid
    df['lat']      = lat
    df['lon']      = lon
    df['depth']    = BUOY_DEPTH_METERS
    df['ssh']      = np.nan
    buoys_list.append(df)

all_buoys_df = cudf.concat(buoys_list, ignore_index=True)

final_cols = ['time', 'point_id', 'lat', 'lon', 'hm0', 'tp', 'mdir',
              'windspeed', 'winddirection', 'depth', 'ssh']

offshore_sel = final_offshore_df[[c for c in final_cols if c in final_offshore_df.columns]].to_pandas()
buoy_sel     = all_buoys_df[[c for c in final_cols if c in all_buoys_df.columns]].to_pandas()

# FIX: Align dtypes column-by-column before concat to prevent cudf type mismatch
for col in offshore_sel.columns:
    if col in buoy_sel.columns:
        try:
            buoy_sel[col] = buoy_sel[col].astype(offshore_sel[col].dtype)
        except (ValueError, TypeError):
            offshore_sel[col] = offshore_sel[col].astype(str)
            buoy_sel[col]     = buoy_sel[col].astype(str)

final_master_pd = (pd.concat([offshore_sel, buoy_sel], ignore_index=True))
final_master_pd['time'] = (
    pd.to_datetime(final_master_pd['time'], format='mixed', utc=True)
    .dt.tz_localize(None)
)
final_master_pd = final_master_pd.sort_values(['point_id', 'time']).reset_index(drop=True)

master_path = os.path.join(OUTPUT_DIR, 'master_dataset_causally_optimized_v5.csv')
final_master_pd.to_csv(master_path, index=False, date_format='%Y-%m-%d %H:%M:%S')
print(f"  ✓ Master dataset saved → {master_path}  shape: {final_master_pd.shape}")

del final_master_pd, final_offshore_df, all_buoys_df, offshore_for_final
free_gpu_memory()

print("\n" + "=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

Installing required libraries...
fatal: destination path 'rapidsai-csp-utils' already exists and is not an empty directory.
Installing RAPIDS remaining 25.10 libraries
Using Python 3.12.12 environment at: /usr
Audited 9 packages in 156ms

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do not run any further installation from the conda based installation methods, as they may cause issues!

        Please ensure that you're pulling from the git repo to remain updated with the latest working install scripts.

        Troubleshooting:
            - If there is an installation failure, please check back on RAPIDSAI owned templates/notebooks to see how to update your personal files.
            - If an installation failure persists when using the latest script, please make an issue on https://github.com/rapidsai-community/rapidsai-csp-utils
        *************************************************

    ✓ ADF table saved.


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/Predictor_Selection_v5_Physical/fig01_acf_tau_max_justification.png

STEP B COMPLETE.

STEP C: PHYSICALLY-WEIGHTED CORRELATIONAL FILTERING
  ✓ Checkpoint found → /content/drive/MyDrive/KBS_Paper/Outputs/Predictor_Selection_v5_Physical/checkpoint_correlation_candidates_v5.csv
  ✓ Top 12 candidates: ['offshore_24', 'offshore_34', 'offshore_56', 'offshore_35', 'offshore_45', 'offshore_25', 'offshore_16', 'offshore_57', 'offshore_46', 'offshore_36', 'offshore_26', 'offshore_79']


,point_id,rank_hm0,rank_tp,rank_mdir_u,rank_mdir_v,total_rank,depth,depth_rank,physically_weighted_rank
0,offshore_24,2.0,1.0,2.0,2.0,7.0,21.0,8.0,15.0
1,offshore_34,1.0,4.0,1.0,1.0,7.0,28.0,11.0,18.0
2,offshore_56,6.0,17.0,3.0,3.0,29.0,14.0,5.0,34.0
3,offshore_35,5.0,11.0,6.0,6.0,28.0,71.0,18.0,46.0
4,offshore_45,3.0,16.0,4.0,4.0,27.0,91.0,22.0,49.0



STEP C COMPLETE.

STEP D: MULTIVARIATE CAUSAL SELECTION (PCMCI+)
  - PCMCI+ input: 68 time series
  - GPU RAM freed; running PCMCI+ on CPU ...
  ✓ Checkpoint found → /content/drive/MyDrive/KBS_Paper/Outputs/Predictor_Selection_v5_Physical/checkpoint_pcmci_results_v5.npz
  - Extracting causal parents + FDR-corrected q-values ...
  ✓ Final 5 predictors: ['offshore_34', 'offshore_56', 'offshore_26', 'offshore_79', 'offshore_46']

STEP D COMPLETE.

STEP E: REPORTING, VISUALIZATION, FINAL DATASET
  - Generating predictor map ...


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/Predictor_Selection_v5_Physical/fig03_predictor_map.png
  - Generating causal graph ...


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/Predictor_Selection_v5_Physical/fig02_causal_graph.png
  ✓ Final predictor IDs saved.
  - Building final master dataset ...
    - Loading offshore CSVs in GPU chunks to prevent OOM ...
      · Batch 1/2 loaded.
      · Batch 2/2 loaded.
    - Loaded 101 unique offshore points.
  ✓ Master dataset saved → /content/drive/MyDrive/KBS_Paper/Outputs/Predictor_Selection_v5_Physical/master_dataset_causally_optimized_v5.csv  shape: (125972, 11)

  PIPELINE COMPLETE
